<a href="https://colab.research.google.com/github/OlenaVN/CREW_notebooks_examples/blob/main/L2_research_write_article.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

You can download the requirements.txt for this course from the workspace of this lab. File --> Open...

# L2: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [ ]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's gpt-3.5-turbo.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the

In [ ]:
import os
from utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [ ]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [ ]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [ ]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

 Define your Tasks, and provide them a description, expected_output and agent.

### Task: Plan

In [ ]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [ ]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [ ]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution.

In [ ]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [ ]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer:
Title: The Latest Trends and Key Players in Artificial Intelligence

Introduction:
- Brief overview of the significance of Artificial Intelligence in today's technology landscape
- Setting the stage for the discussion on the latest trends and key players in the field

Key Points:
1. Latest Trends:
- Machine Learning advancements
- Natural Language Processing applications
- AI in healthcare and medicine
- Autonomous vehicles and robotics
- AI ethics and bias concerns

2. Key Players:
- Google DeepMind
- IBM Wat

I now can give a great answer.

Final Answer:
# The Latest Trends and Key Players in Artificial Intelligence

## Introduction
Artificial Intelligence (AI) has become a central component of today's technology landscape, revolutionizing industries and daily life. From personalized recommendations on streaming services to self-driving cars, AI is reshaping the way we interact with technology. In this blog post, we will delve into the latest trends and key players driving the advancements in AI, exploring the key developments that are shaping the future of this rapidly evolving field.

## Latest Trends
One of the most significant trends in AI is the advancements in Machine Learning, allowing algorithms to learn from data and improve their performance over time. This has led to breakthroughs in various applications, from image recognition to natural language understanding. Natural Language Processing (NLP) is another key trend, enabling machines to understand and generate human language, po

- Display the results of your execution as markdown in the notebook.

In [ ]:
from IPython.display import Markdown
Markdown(result)

# The Latest Trends and Key Players in Artificial Intelligence

## Introduction
Artificial Intelligence (AI) has become a central component of today's technology landscape, revolutionizing industries and daily life. From personalized recommendations on streaming services to self-driving cars, AI is reshaping the way we interact with technology. In this blog post, we will delve into the latest trends and key players driving the advancements in AI, exploring the key developments that are shaping the future of this rapidly evolving field.

## Latest Trends
One of the most significant trends in AI is the advancements in Machine Learning, allowing algorithms to learn from data and improve their performance over time. This has led to breakthroughs in various applications, from image recognition to natural language understanding. Natural Language Processing (NLP) is another key trend, enabling machines to understand and generate human language, powering virtual assistants and language translation tools. In the healthcare and medicine sector, AI is being used to analyze medical images, diagnose diseases, and personalize treatment plans, improving patient outcomes. Autonomous vehicles and robotics are also benefiting from AI technologies, with self-driving cars and intelligent robots becoming a reality.

Ethical concerns and biases in AI have also come to the forefront, with discussions on the responsible use of AI and the need to address biases in algorithms. As AI becomes more integrated into our daily lives, these ethical considerations are crucial to ensure fair and transparent use of AI technologies.

## Key Players
Several key players are driving innovation in the field of AI, with companies like Google DeepMind, IBM Watson, and OpenAI leading the way. Google DeepMind is known for its advancements in deep learning and reinforcement learning, pushing the boundaries of AI research. IBM Watson is a pioneer in cognitive computing, with applications in healthcare, finance, and customer service. OpenAI focuses on developing safe and beneficial AI technologies, advocating for ethical AI research and deployment. NVIDIA is a key player in providing hardware solutions for AI applications, powering deep learning algorithms with their GPUs. Microsoft Azure Cognitive Services offer a range of AI tools and services for developers, enabling them to integrate AI capabilities into their applications.

## Noteworthy News
Recent breakthroughs in AI research have included advancements in unsupervised learning, reinforcement learning, and generative models. Industry partnerships and collaborations are also on the rise, with companies working together to accelerate AI innovation and adoption. Regulatory developments, such as the implementation of AI guidelines and standards, are shaping the future of AI technologies, ensuring responsible and ethical use of AI in various domains.

As the AI landscape continues to evolve, staying informed about the latest trends and key players is essential for technology enthusiasts, industry professionals, and decision-makers. By following reputable sources and engaging with AI news and updates, readers can stay ahead of the curve and contribute to the responsible development and deployment of AI technologies.

## Conclusion
In conclusion, Artificial Intelligence is at the forefront of technological innovation, with the latest trends and key players driving the advancements in this field. From Machine Learning to NLP, AI in healthcare to autonomous vehicles, the possibilities with AI are limitless. By staying informed and engaging with AI developments, readers can be part of the exciting journey towards a more intelligent future powered by AI technologies.

Remember to subscribe to newsletters and follow reputable sources for the latest AI news and updates. Share your thoughts and experiences with AI technologies to join the conversation and contribute to the ongoing evolution of Artificial Intelligence. Let's embrace the potential of AI while navigating the ethical considerations to ensure a bright and responsible future for AI technologies.

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [ ]:
topic = "All information about Advantix for dogs"
result = crew.kickoff(inputs={"topic": topic})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on All information about Advantix for dogs.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer: 

Content Plan: All Information about Advantix for Dogs

Introduction:
- Brief overview of Advantix for dogs
- Importance of flea, tick, and mosquito prevention for dogs
- Introduce key points to be covered in the article

Key Points:
1. Latest Trends:
- Advancements in Advantix formula for better protection
- New packaging designs for easier application
- Increase in popularity among dog owners for its effectiveness

2. Key Players:
- Bayer Animal Health as the manufacturer of Advantix
- Veter

I now can give a great answer

Final Answer:

# All You Need to Know About Advantix for Dogs

Advantix for dogs is a popular and effective solution for flea, tick, and mosquito prevention in dogs. As a pet owner, ensuring the health and well-being of your furry friend is crucial, and Advantix offers a comprehensive approach to protecting your dog from these pesky parasites. In this article, we will delve into the latest trends in Advantix, the key players involved in its development and distribution, as well as any noteworthy news regarding its use. Whether you're a seasoned pet parent or a first-time dog owner, understanding the ins and outs of Advantix can help you make informed decisions about your dog's care.

Advantix has made strides in advancing its formula to provide better protection against fleas, ticks, and mosquitoes. With new packaging designs that make application easier and more convenient, dog owners are finding Advantix to be a go-to solution for parasite prevention. T

In [ ]:
Markdown(result)

# All You Need to Know About Advantix for Dogs

Advantix for dogs is a popular and effective solution for flea, tick, and mosquito prevention in dogs. As a pet owner, ensuring the health and well-being of your furry friend is crucial, and Advantix offers a comprehensive approach to protecting your dog from these pesky parasites. In this article, we will delve into the latest trends in Advantix, the key players involved in its development and distribution, as well as any noteworthy news regarding its use. Whether you're a seasoned pet parent or a first-time dog owner, understanding the ins and outs of Advantix can help you make informed decisions about your dog's care.

Advantix has made strides in advancing its formula to provide better protection against fleas, ticks, and mosquitoes. With new packaging designs that make application easier and more convenient, dog owners are finding Advantix to be a go-to solution for parasite prevention. The increase in popularity among dog owners is a testament to Advantix's effectiveness in keeping dogs healthy and free from infestations.

Bayer Animal Health stands as the manufacturer behind Advantix, a trusted name in the pet care industry. Veterinary professionals across the board recommend Advantix for dogs, highlighting its efficacy and safety profile. Online retailers have also made Advantix readily accessible to pet owners, ensuring that this top-tier product is within reach for those seeking reliable parasite prevention for their dogs.

Recent studies have shed light on the long-term effects of Advantix on dogs, providing valuable insights into its safety and efficacy. Updates on any recalls or safety concerns related to Advantix are crucial for pet owners to stay informed and make educated decisions about their dog's care. Success stories from dog owners who have used Advantix effectively serve as testimonials to its effectiveness in protecting dogs from parasites.

In conclusion, Advantix for dogs offers a comprehensive solution for flea, tick, and mosquito prevention, backed by advancements in formula, support from key players like Bayer Animal Health, and positive feedback from pet owners and veterinarians alike. By staying informed on the latest trends and news surrounding Advantix, dog owners can make confident decisions about their dog's health and well-being. Remember, always consult with your veterinarian before using Advantix on your dog and share your experiences with us in the comments below.

Remember, the health and safety of your furry companion are paramount, and Advantix is here to help you keep your dog protected and parasite-free. Thank you for reading and stay tuned for more insightful articles on pet care products and practices.

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).